# 🗄️ Course 4 — Joining Data in SQL

> **Platform:** DataCamp | **Track:** Associate Data Analyst in SQL  
> **Tool:** PostgreSQL | **Database:** Countries (cities, countries, economies, populations, languages, currencies)

---

## 📋 About This Course

This course covers all major SQL join types and set operations. Using a countries database with information on cities, economies, populations and languages, it progresses from basic INNER JOINs through OUTER, CROSS and SELF JOINs, set theory operations, and advanced subqueries.

---

## 📚 Table of Contents

| Chapter | Topic |
|---------|-------|
| Chapter 1 | Inner Joins |
| Chapter 2 | Outer Joins, Cross Joins & Self Joins |
| Chapter 3 | Set Theory for SQL Joins |
| Chapter 4 | Subqueries |

---

## 📌 Chapter 1 — Inner Joins

---

### Your First Join
Join `cities` and `countries` using `INNER JOIN` on `country_code = code`. Progress from selecting all columns to specific aliased fields.

In [ ]:
SELECT * 
FROM cities;

In [ ]:
SELECT * 
FROM cities
INNER JOIN countries
ON cities.country_code = countries.code;

In [ ]:
SELECT cities.name AS city, countries.name AS country, countries.region
FROM cities
INNER JOIN countries
ON cities.country_code = countries.code;

### Joining with Aliased Tables
Join `countries AS c` with `economies AS e` on `code`. Select `country_code`, `name`, `year`, and `inflation_rate`.

In [ ]:
SELECT c.code AS country_code, name, year, inflation_rate
FROM countries AS c
INNER JOIN economies AS e
ON c.code = e.code;

### USING in Action
Use the `USING` clause to simplify joins when both tables share the same column name.

In [ ]:
SELECT c.name AS country, l.name AS language, official
FROM countries AS c
INNER JOIN languages AS l
USING(code);

### Relationships in Our Database
**Q:** What best describes `code` in `countries` vs `country_code` in `cities`?  
**A:** One-to-many relationship.

**Q:** What is the relationship between `countries` and `languages`?  
**A:** Many-to-many relationship.

### Inspecting a Relationship
Find all countries that speak 'Bhojpuri' using INNER JOIN + WHERE.

In [ ]:
SELECT c.name AS country, l.name AS language
FROM countries AS c
INNER JOIN languages AS l
USING(code)
WHERE l.name = 'Bhojpuri';
-- Answer: 2 countries

### Joining Multiple Tables
Chain two INNER JOINs: `countries` → `populations` → `economies` to get country name, year, fertility rate and unemployment rate.

In [ ]:
SELECT name, e.year, fertility_rate, e.unemployment_rate
FROM countries AS c
INNER JOIN populations AS p
ON c.code = p.country_code
INNER JOIN economies AS e
ON c.code = e.code
AND p.year = e.year;

### Checking Multi-Table Joins
Fix a multi-table join that produces duplicate rows by also joining on `year`.

In [ ]:
SELECT name, e.year, fertility_rate, unemployment_rate
FROM countries AS c
INNER JOIN populations AS p
ON c.code = p.country_code
INNER JOIN economies AS e
ON c.code = e.code
	AND p.year = e.year;

---

## 📌 Chapter 2 — Outer Joins, Cross Joins & Self Joins

---

### LEFT JOIN vs INNER JOIN
Compare INNER JOIN and LEFT JOIN between `cities` and `countries` to understand how unmatched records are handled.

In [ ]:
-- INNER JOIN
SELECT c1.name AS city, code, c2.name AS country, region, city_proper_pop
FROM cities AS c1
INNER JOIN countries AS c2
ON c1.country_code = c2.code
ORDER BY code DESC;

In [ ]:
-- LEFT JOIN
SELECT c1.name AS city, code, c2.name AS country, region, city_proper_pop
FROM cities AS c1
LEFT JOIN countries AS c2
ON c1.country_code = c2.code
ORDER BY code DESC;

### Building on LEFT JOIN
Calculate average GDP per capita by region in 2010 using LEFT JOIN + AVG() + GROUP BY.

In [ ]:
SELECT region, AVG(gdp_percapita) AS avg_gdp
FROM countries AS c
LEFT JOIN economies AS e
USING(code)
WHERE year = 2010
GROUP BY region
ORDER BY avg_gdp DESC
LIMIT 10;

### RIGHT JOIN
Rewrite a LEFT JOIN as a RIGHT JOIN by swapping the table order — produces identical results.

In [ ]:
SELECT countries.name AS country, languages.name AS language, percent
FROM languages
RIGHT JOIN countries
USING(code)
ORDER BY language;

### Comparing Joins: FULL, LEFT, INNER
Compare three join types filtering for North America or NULL country names.

In [ ]:
-- FULL JOIN
SELECT name AS country, code, region, basic_unit
FROM countries
FULL JOIN currencies USING (code)
WHERE region = 'North America' OR region IS NULL
ORDER BY region;

In [ ]:
-- LEFT JOIN
SELECT name AS country, code, region, basic_unit
FROM countries
LEFT JOIN currencies USING (code)
WHERE region = 'North America' OR name IS NULL
ORDER BY region;

In [ ]:
-- INNER JOIN
SELECT name AS country, code, region, basic_unit
FROM countries
INNER JOIN currencies USING (code)
WHERE region = 'North America' OR name IS NULL
ORDER BY region;

### Chaining FULL JOINs
Chain two FULL JOINs to combine `countries`, `languages`, and `currencies` for Melanesia and Micronesia regions.

In [ ]:
SELECT c1.name AS country, region, l.name AS language, basic_unit, frac_unit
FROM countries AS c1
FULL JOIN languages AS l USING (code)
FULL JOIN currencies AS c2 USING (code)
WHERE region LIKE 'M%esia';

### INNER JOIN vs CROSS JOIN: Histories and Languages
Compare languages currently spoken in Pakistan and India (INNER JOIN) vs all possible combinations (CROSS JOIN).

In [ ]:
-- INNER JOIN: current languages
SELECT c.name AS country, l.name AS language
FROM countries AS c
INNER JOIN languages AS l USING(code)
WHERE c.code IN ('PAK','IND') AND l.code IN ('PAK','IND');

In [ ]:
-- CROSS JOIN: all possible combinations
SELECT c.name AS country, l.name AS language
FROM countries AS c
CROSS JOIN languages AS l
WHERE c.code IN ('PAK','IND') AND l.code IN ('PAK','IND');

### Choosing Your Join
Find the 5 countries with the lowest life expectancy in 2010.

In [ ]:
SELECT c.name AS country, region, life_expectancy AS life_exp
FROM countries AS c
INNER JOIN populations AS p
ON c.code = p.country_code
WHERE year = 2010
ORDER BY life_expectancy ASC
LIMIT 5;

### Self Join: Comparing a Country to Itself
Join `populations` with itself to compare population sizes in 2010 vs 2015.

In [ ]:
SELECT p1.country_code, p1.size AS size2010, p2.size AS size2015
FROM populations AS p1
INNER JOIN populations AS p2
ON p1.country_code = p2.country_code
WHERE p1.year = 2010
	AND p1.year = p2.year - 5;

---

## 📌 Chapter 3 — Set Theory for SQL Joins

---

### UNION vs UNION ALL
**Q:** `SELECT * FROM languages UNION SELECT * FROM currencies`  
**A:** SQL error — tables have different number of fields.

**Q:** `SELECT code FROM languages UNION ALL SELECT code FROM currencies`  
**A:** Unordered list including duplicates.

**Q:** `SELECT code FROM languages UNION SELECT curr_id FROM currencies`  
**A:** SQL error — different data types.

### Comparing Global Economies
Stack all records from `economies2015` and `economies2019` without duplicates using UNION.

In [ ]:
SELECT * FROM economies2015
UNION
SELECT * FROM economies2019
ORDER BY code, year;

### Comparing Two Set Operations
Return all pairs of country code and year from `economies` and `populations`, with and without duplicates.

In [ ]:
-- Without duplicates
SELECT code AS country_code, year FROM economies
UNION
SELECT country_code, year FROM populations
ORDER BY country_code, year;

In [ ]:
-- Including duplicates
SELECT code, year FROM economies
UNION ALL
SELECT country_code, year FROM populations
ORDER BY code, year;

### INTERSECT
Return all city names that are also country names.

In [ ]:
SELECT name FROM cities
INTERSECT
SELECT name FROM countries;

### EXCEPT
Return all city names that do NOT match any country name.

In [ ]:
SELECT name FROM cities
EXCEPT
SELECT name FROM countries
ORDER BY name;

---

## 📌 Chapter 4 — Subqueries

---

### Semi Join
Find languages spoken in the Middle East using a subquery inside WHERE (semi join).

In [ ]:
SELECT DISTINCT name
FROM languages
WHERE code IN
    (SELECT code
     FROM countries
     WHERE region = 'Middle East')
ORDER BY name;

### Anti Join: Diagnosing Problems
Find Oceanian countries excluded from a currencies INNER JOIN using a NOT IN anti join.

In [ ]:
SELECT code, name
FROM countries
WHERE continent = 'Oceania'
  AND code NOT IN
    (SELECT code FROM currencies);

### Subquery Inside WHERE
Find countries with life expectancy above 1.15× the 2015 average.

In [ ]:
SELECT *
FROM populations
WHERE year = 2015
  AND life_expectancy > 1.15 *
  (SELECT AVG(life_expectancy)
   FROM populations
   WHERE year = 2015);

### WHERE Do People Live?
Return capital cities ordered by urban area population using a subquery filter.

In [ ]:
SELECT name, country_code, urbanarea_pop
FROM cities
WHERE name IN
    (SELECT capital FROM countries)
ORDER BY urbanarea_pop DESC;

### Subquery Inside SELECT
Find the 9 countries with the most cities — using LEFT JOIN + GROUP BY, then rewrite with a subquery inside SELECT.

In [ ]:
-- Using LEFT JOIN
SELECT countries.name AS country, COUNT(*) AS cities_num
FROM countries
LEFT JOIN cities ON countries.code = cities.country_code
GROUP BY country
ORDER BY cities_num DESC, country ASC
LIMIT 9;

In [ ]:
-- Using subquery inside SELECT
SELECT countries.name AS country,
  (SELECT COUNT(*)
   FROM cities
   WHERE cities.country_code = countries.code) AS cities_num
FROM countries
ORDER BY cities_num DESC, country
LIMIT 9;

### Subquery Inside FROM
Count languages per country and join with `local_name` from countries using a subquery inside FROM.

In [ ]:
SELECT countries.local_name, sub.lang_num
FROM countries,
  (SELECT code, COUNT(*) AS lang_num
   FROM languages
   GROUP BY code) AS sub
WHERE countries.code = sub.code
ORDER BY lang_num DESC;

### Subquery Challenge
Find inflation and unemployment rates in 2015 for countries with 'Republic' or 'Monarchy' government forms.

In [ ]:
SELECT code, inflation_rate, unemployment_rate
FROM economies
WHERE year = 2015
  AND code IN
    (SELECT code FROM countries
     WHERE gov_form LIKE '%Monarchy%' OR gov_form LIKE '%Republic%')
ORDER BY inflation_rate;

### Final Challenge
Find the top 10 capital cities in Europe and the Americas by city population as a % of metro area population.

In [ ]:
SELECT name, country_code, city_proper_pop, metroarea_pop,
       (city_proper_pop / metroarea_pop * 100) AS city_perc
FROM cities
WHERE cities.name IN
    (SELECT capital FROM countries
     WHERE continent LIKE 'Europe' OR continent LIKE '%America')
  AND metroarea_pop IS NOT NULL
ORDER BY city_perc DESC
LIMIT 10;